# 12_generic_augmentation_control.ipynb

Task 5.6, đối chứng augmentation tổng quát. Huấn luyện lại Swin-Tiny (reference model) chỉ với TrivialAugment, không có phép làm hỏng nhắm vào benchmark, rồi so với reference model và với model huấn luyện lại bằng augmentation suy giảm của task 4.5. Mục đích là tách phần cải thiện do augmentation nói chung khỏi phần do augmentation nhắm đúng các họ suy giảm.

Cùng giao thức với notebook 07: cùng model, optimizer, scheduler, epoch, patience, seed, split và quy tắc chọn checkpoint (validation hỗn hợp, ảnh sạch cộng mỗi họ ở mức 2). Khác một điểm: bước augmentation thêm vào là TrivialAugment thay cho suy giảm ngẫu nhiên. Các giá trị khóa trong `configs/train_swin_tiny_trivialaugment.yaml` trước khi chạy.

Đánh giá trên validation: ảnh sạch, 42 điều kiện của 14 họ đã thấy ở benchmark 4.1, và 18 điều kiện của 6 họ chưa thấy ở task 5.5. Tập test không mở ở notebook này, việc đo trên test thuộc P6.1 với cấu hình đã khóa.

## 1. Môi trường

Internet để On. Accelerator chọn GPU (T4). Input bắt buộc: dataset `cthng123/durian-leaf-disease-dataset`. Input tùy chọn: dataset chứa `swin_tiny_seed42.pt` và `swin_tiny_aug_seed42.pt`. Có hai checkpoint này thì notebook đo thêm hai model đó trên 18 điều kiện chưa thấy để so ghép cặp theo từng ảnh, không có thì phần chưa thấy chỉ so điểm macro F1.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "generic-augmentation-control"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Cấu hình và dữ liệu

Kiểm tra GPU, tạo loader train có TrivialAugment và validation hỗn hợp (tạo sẵn một lần, khoảng một đến hai phút). Thiếu GPU thì dừng ngay.

In [ ]:
import json
import os
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from torch import nn

from src.corruption import build_loader, load_conditions, load_seed
from src.evaluation import plot_corruption_examples, plot_history
from src.evaluation.condition_predictions import evaluate_with_predictions, save_predictions
from src.training.loop import evaluate_loader
from src.training.run import ROOT, load_configs, print_run_report, resource_name, save_run_artifacts
from src.training.run_generic_aug import amend_record, prepare_loaders

FIG = REPO_DIR / "outputs/figures/generic_augmentation"
RES = REPO_DIR / "outputs/results/generic_augmentation"
REP = REPO_DIR / "reports/generic_augmentation"
for folder in (FIG, RES, REP):
    folder.mkdir(parents=True, exist_ok=True)


def show(path):
    display(Image(filename=str(path)))


CONFIG = "configs/train_swin_tiny_trivialaugment.yaml"
SEEN_CONFIG = "configs/corruption.yaml"
UNSEEN_CONFIG = "configs/unseen_corruption.yaml"
train_cfg, pipeline_cfg = load_configs(CONFIG)
classes = list(pipeline_cfg["classes"])
device, loaders, augmentation = prepare_loaders(train_cfg, pipeline_cfg)

print(f"Thiết bị: {device} ({resource_name(device)})")
print(f"Seed: {pipeline_cfg['seed']}")
print(f"Augmentation: {augmentation}")
print(f"Ảnh train: {len(loaders['train'].dataset)}")
print(f"Validation hỗn hợp: {len(loaders['val'].dataset)} ảnh (sạch cộng 14 họ ở mức {train_cfg['validation']['severity']})")
print(f"Optimizer: {train_cfg['optimizer']}, lr {train_cfg['learning_rate']}, {train_cfg['scheduler']}, tối đa {train_cfg['epochs']} epoch, patience {train_cfg['patience']}")
print("Tập test: không mở")

### Ảnh augmentation mẫu

Một ảnh validation qua TrivialAugment nhiều lần. Mỗi lần áp đúng một phép với cường độ ngẫu nhiên (màu, độ sáng, tương phản, độ nét, cắt xéo, dịch chuyển, đảo màu cục bộ). Các phép này là biến đổi ảnh tự nhiên, không mô phỏng cơ chế suy giảm của benchmark, tuy một số phép như độ sáng, tương phản và độ nét có thể trùng vùng giá trị với vài họ của benchmark. Phần này được xét ở kết luận.

In [ ]:
from PIL import Image as PILImage

from src.data.splits import get_split

row = get_split("val").iloc[0]
pil = PILImage.open(REPO_DIR / "datasets/durian-ldd" / row["image_path"]).convert("RGB")
torch.manual_seed(0)
rows = [(f"lần {index + 1}", [("gốc", np.asarray(pil) / 255.0)] + [("TrivialAugment", np.asarray(augmentation(pil)) / 255.0) for _ in range(5)]) for index in range(3)]
plot_corruption_examples(rows, FIG / "trivialaugment_examples.png", dpi=100)
show(FIG / "trivialaugment_examples.png")

## 3. Model

Swin-Tiny (`swin_tiny_patch4_window7_224`) từ timm, trọng số ImageNet, fine-tune toàn bộ mạng, 5 lớp. Giống hệt baseline và task 4.5.

In [ ]:
from src.models.build import build_model
from src.training.loop import trainable_parameter_count

model = build_model(train_cfg, num_classes=int(pipeline_cfg["num_classes"]))
print(f"Backbone: {train_cfg['library']} {train_cfg['backbone']}")
print(f"Số tham số: {trainable_parameter_count(model)}")

## 4. Train

Mỗi epoch in loss, macro F1 và balanced accuracy trên validation hỗn hợp. Dòng có chữ `checkpoint` là epoch đang được giữ.

In [ ]:
from src.training.loop import fit_model

result = fit_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    class_names=classes,
    train_cfg=train_cfg,
    device=device,
)
print(f"Epoch được chọn: {result['best_epoch']}")
print(f"Thời gian train (giây): {result['elapsed_seconds']:.1f}")

## 5. Kết quả validation hỗn hợp

Lưu checkpoint tốt nhất, file kết quả và hình, cùng định dạng với các baseline. Các chỉ số `val_*` tính trên tập hỗn hợp. Điểm trên validation sạch (không dùng để chọn) được ghi riêng.

In [ ]:
paths = save_run_artifacts(result, pipeline_cfg, train_cfg, device, ROOT / "outputs")
clean_val = evaluate_loader(model, build_loader(pipeline_cfg, "val"), nn.CrossEntropyLoss(), device, classes)
record = amend_record(paths, train_cfg, augmentation, clean_val, len(loaders["val"].dataset))
print_run_report(result, paths, resource_name(device))
print(f"Validation sạch (không dùng để chọn checkpoint): macro F1 {clean_val['macro_f1']:.4f}, balanced accuracy {clean_val['balanced_accuracy']:.4f}")

best = result["best_val"]
display(pd.DataFrame([{
    "best_epoch": result["best_epoch"],
    "mixed_val_macro_f1": best["macro_f1"],
    "mixed_val_balanced_accuracy": best["balanced_accuracy"],
    "clean_val_macro_f1": clean_val["macro_f1"],
    "parameters": result["trainable_parameters"],
    "seconds": round(result["elapsed_seconds"], 1),
}]))
display(pd.DataFrame([{"class_name": name, "recall (validation sạch)": value} for name, value in clean_val["per_class_recall"].items()]))
display(pd.DataFrame(result["history"]))
show(paths["history"])
show(paths["confusion_matrix"])

## 6. Validation: ảnh sạch và 42 điều kiện đã thấy

Cùng đường đi ảnh với benchmark 4.1, nên so được từng ảnh với reference model và model augmentation suy giảm. Lưu dự đoán và xác suất từng ảnh.

In [ ]:
seen_seed = load_seed(SEEN_CONFIG)
seen_conditions = load_conditions(SEEN_CONFIG, "val")
CONFIG_ID = train_cfg["model_name"]
seen_result = evaluate_with_predictions(model, pipeline_cfg, "val", seen_conditions, device, seen_seed, "retrained", CONFIG_ID, progress=lambda _: None)
seen_frame = seen_result["frame"]
seen_frame.to_csv(RES / "generic_augmentation_val_conditions.csv", index=False)
save_predictions(seen_result, RES / "generic_augmentation_val_predictions.npz")
print(f"Macro F1 validation sạch: {float(seen_frame.loc[seen_frame['condition'] == 'clean', 'macro_f1'].iloc[0]):.4f}")
table = seen_frame[seen_frame["condition"] != "clean"].pivot(index="family", columns="severity", values="macro_f1")
display(table.round(3).assign(trung_binh=table.mean(axis=1).round(3)))

## 7. Validation: 18 điều kiện của 6 họ chưa thấy

Các họ của task 5.5 không xuất hiện ở bất kỳ augmentation nào. Config đã khóa. Họ gần (`near`) cùng cơ chế với một họ đã thấy, họ xa (`far`) có cơ chế mới.

In [ ]:
unseen_seed = load_seed(UNSEEN_CONFIG)
unseen_conditions = load_conditions(UNSEEN_CONFIG, "val")
unseen_names = list(unseen_conditions)
unseen_groups = {family: spec["group"] for family, spec in yaml.safe_load((REPO_DIR / UNSEEN_CONFIG).read_text(encoding="utf-8"))["families"].items()}
unseen_result = evaluate_with_predictions(model, pipeline_cfg, "val", unseen_conditions, device, unseen_seed, "retrained", CONFIG_ID, progress=lambda _: None)
unseen_frame = unseen_result["frame"]
unseen_frame.to_csv(RES / "generic_augmentation_unseen_val_conditions.csv", index=False)
save_predictions(unseen_result, RES / "generic_augmentation_unseen_val_predictions.npz")
table = unseen_frame[unseen_frame["condition"] != "clean"].pivot(index="family", columns="severity", values="macro_f1")
display(table.round(3).assign(trung_binh=table.mean(axis=1).round(3), nhom=[unseen_groups[family] for family in table.index]))

### Đo thêm hai model so sánh trên họ chưa thấy (tùy chọn)

Nếu tìm thấy checkpoint của reference model và của model augmentation suy giảm trong `/kaggle/input`, đo chúng trên cùng 18 điều kiện. Không thấy thì bỏ qua, phần so sánh chưa thấy khi đó dùng điểm reference đã lưu ở task 5.5.

In [ ]:
from src.evaluation.reference import collect_checkpoint, load_trained_model

COMPARATORS = {"reference": ("swin_tiny", "configs/train_swin_tiny.yaml"), "degradation_aug": ("swin_tiny_aug", "configs/train_swin_tiny_aug.yaml")}
unseen_extra = {}
for label, (file_stem, config_path) in COMPARATORS.items():
    try:
        checkpoint = collect_checkpoint("/kaggle/input", f"{file_stem}_seed42.pt", Path("/kaggle/working/checkpoints"))
    except FileNotFoundError:
        print(f"{label}: không thấy {file_stem}_seed42.pt, bỏ qua")
        continue
    other_cfg, _ = load_configs(config_path)
    other = load_trained_model(other_cfg, checkpoint, len(classes), device)
    outcome = evaluate_with_predictions(other, pipeline_cfg, "val", unseen_conditions, device, unseen_seed, "retrained", file_stem, progress=lambda _: None)
    save_predictions(outcome, RES / f"unseen_val_predictions_{file_stem}.npz")
    unseen_extra[label] = (outcome["labels"], file_stem)
    print(f"{label}: macro F1 sạch {float(outcome['frame'].loc[outcome['frame']['condition'] == 'clean', 'macro_f1'].iloc[0]):.4f}")
    del other
print("Đã đo:", ", ".join(unseen_extra) or "(không có)")

## 8. So sánh trên 42 điều kiện đã thấy

Ba cách huấn luyện trên cùng 444 ảnh validation: reference (không augmentation thêm), TrivialAugment (notebook này) và augmentation suy giảm (task 4.5). Mục tiêu là macro F1 trung bình 42 điều kiện. Khoảng tin cậy là bootstrap ghép cặp theo ảnh với 1.000 lần lấy mẫu, McNemar chính xác từng điều kiện có hiệu chỉnh Holm trên 42 điều kiện.

In [ ]:
from src.evaluation.control_compare import (
    check_same_images, condition_comparison, family_scores, reference_matrix, retrained_matrix, summarize_methods, verdict,
)
from src.evaluation.control_plots import plot_delta_heatmap, plot_family_bars, plot_levels, plot_objective, plot_scatter_methods

seen_names = list(seen_conditions)
labels_ref, reference = reference_matrix(REPO_DIR / "reports/sensitivity/sensitivity_predictions.npz", seen_names)
labels_aug, degradation = retrained_matrix(REPO_DIR / "reports/augmentation/augmentation_val_predictions.npz", "swin_tiny_aug", seen_names)
labels_ta, trivial = retrained_matrix(RES / "generic_augmentation_val_predictions.npz", CONFIG_ID, seen_names)
labels = check_same_images({"reference": labels_ref, "degradation_aug": labels_aug, "trivialaugment": labels_ta})
matrices = {"reference": reference, "trivialaugment": trivial, "degradation_aug": degradation}

summary = summarize_methods(labels, matrices, seen_names, len(classes), base="reference", resamples=1000, seed=42)
summary["so với reference"] = [verdict(r.delta_vs_base, r.delta_low, r.delta_high) if r.method != "reference" else "" for r in summary.itertuples()]
summary.to_csv(REP / "seen_summary_val.csv", index=False)
display(summary.round(4))

plot_objective(summary, FIG / "seen_objective.png", "42 điều kiện đã thấy, validation")
show(FIG / "seen_objective.png")
plot_levels(summary, FIG / "seen_levels.png", "Macro F1 theo mức suy giảm, 14 họ đã thấy")
show(FIG / "seen_levels.png")

### Phần cải thiện tách theo nguồn

Với reference làm gốc, hiệu của augmentation suy giảm gồm hai phần: phần TrivialAugment đã đạt được (augmentation nói chung) và phần còn lại do augmentation nhắm đúng các họ. Bảng dưới có hiệu từng cặp kèm khoảng tin cậy ghép cặp.

In [ ]:
from src.evaluation.control_compare import bootstrap_indices, objective_samples, _interval

indices = bootstrap_indices(len(labels), 1000, 42)
samples = {name: objective_samples(labels, matrix, len(classes), indices) for name, matrix in matrices.items()}
point = summary.set_index("method")["objective"]
pairs = [
    ("TrivialAugment trừ reference", "trivialaugment", "reference"),
    ("Suy giảm trừ reference", "degradation_aug", "reference"),
    ("Suy giảm trừ TrivialAugment", "degradation_aug", "trivialaugment"),
]
decomposition = []
for text, a, b in pairs:
    low, high = _interval(samples[a] - samples[b])
    decomposition.append({"so sánh": text, "hiệu": point[a] - point[b], "ci_low": low, "ci_high": high, "kết luận": verdict(point[a] - point[b], low, high)})
decomposition = pd.DataFrame(decomposition)
decomposition["tỉ lệ so với tổng cải thiện"] = decomposition["hiệu"] / decomposition.loc[1, "hiệu"]
decomposition.to_csv(REP / "seen_decomposition_val.csv", index=False)
display(decomposition.round(4))

### Theo họ và theo mức

Biểu đồ cột là macro F1 trung bình ba mức của từng họ. Ma trận nhiệt là hiệu giữa hai cách theo họ và mức, dấu sao là ô có p Holm dưới 0,05 ở McNemar. Đọc theo hàng để thấy họ nào TrivialAugment đã đủ, họ nào cần augmentation nhắm đúng.

In [ ]:
by_family = family_scores(labels, matrices, seen_names, len(classes))
by_family.to_csv(REP / "seen_family_val.csv")
plot_family_bars(by_family, FIG / "seen_family.png", "Macro F1 theo họ, trung bình ba mức", "Macro F1")
show(FIG / "seen_family.png")

ta_vs_ref = condition_comparison(labels, reference, trivial, seen_names, len(classes))
aug_vs_ta = condition_comparison(labels, trivial, degradation, seen_names, len(classes))
ta_vs_ref.to_csv(REP / "seen_conditions_trivialaugment_vs_reference.csv", index=False)
aug_vs_ta.to_csv(REP / "seen_conditions_degradation_vs_trivialaugment.csv", index=False)

plot_delta_heatmap(ta_vs_ref, FIG / "seen_delta_ta_vs_ref.png", "TrivialAugment trừ reference")
show(FIG / "seen_delta_ta_vs_ref.png")
plot_delta_heatmap(aug_vs_ta, FIG / "seen_delta_aug_vs_ta.png", "Augmentation suy giảm trừ TrivialAugment")
show(FIG / "seen_delta_aug_vs_ta.png")
plot_scatter_methods(aug_vs_ta, FIG / "seen_scatter.png", "Macro F1 của TrivialAugment", "Macro F1 của augmentation suy giảm", "Mỗi điểm là một điều kiện")
show(FIG / "seen_scatter.png")

for text, frame in (("TrivialAugment so với reference", ta_vs_ref), ("Augmentation suy giảm so với TrivialAugment", aug_vs_ta)):
    print(f"{text}: tốt hơn có ý nghĩa {int(((frame['delta'] > 0) & (frame['p_holm'] < 0.05)).sum())}, "
          f"kém hơn có ý nghĩa {int(((frame['delta'] < 0) & (frame['p_holm'] < 0.05)).sum())}, "
          f"không phân biệt được {int((frame['p_holm'] >= 0.05).sum())} trong {len(frame)} điều kiện")

### Ảnh sạch và điểm đánh đổi

Một cách augmentation chỉ đáng dùng nếu không làm giảm đáng kể ảnh sạch. Bảng so macro F1 ảnh sạch và số ảnh sạch đổi dự đoán so với reference.

In [ ]:
clean_rows = []
for method, matrix in matrices.items():
    clean_rows.append({
        "method": method,
        "clean_macro_f1": summary.set_index("method").loc[method, "clean_f1"],
        "ảnh sạch đổi dự đoán so với reference": int((matrix[0] != reference[0]).sum()),
        "đúng→sai": int(((reference[0] == labels) & (matrix[0] != labels)).sum()),
        "sai→đúng": int(((reference[0] != labels) & (matrix[0] == labels)).sum()),
    })
clean_table = pd.DataFrame(clean_rows)
clean_table.to_csv(REP / "seen_clean_val.csv", index=False)
display(clean_table.round(4))

## 9. So sánh trên 18 điều kiện chưa thấy

Câu hỏi: augmentation nào chuyển được sang họ suy giảm chưa từng gặp. Họ gần và họ xa báo cáo riêng, vì họ gần có thể hưởng lợi từ họ đã thấy còn họ xa thì không.

In [ ]:
unseen_labels, unseen_ta = retrained_matrix(RES / "generic_augmentation_unseen_val_predictions.npz", CONFIG_ID, unseen_names)
unseen_matrices = {"trivialaugment": unseen_ta}
labels_by_method = {"trivialaugment": unseen_labels}
for label, (_, file_stem) in unseen_extra.items():
    other_labels, other_matrix = retrained_matrix(RES / f"unseen_val_predictions_{file_stem}.npz", file_stem, unseen_names)
    unseen_matrices[label] = other_matrix
    labels_by_method[label] = other_labels
unseen_labels = check_same_images(labels_by_method)
order = [name for name in ("reference", "trivialaugment", "degradation_aug") if name in unseen_matrices]
unseen_matrices = {name: unseen_matrices[name] for name in order}
paired = "reference" in unseen_matrices

if paired:
    unseen_summary = summarize_methods(unseen_labels, unseen_matrices, unseen_names, len(classes), base="reference", resamples=1000, seed=42)
    unseen_summary["so với reference"] = [verdict(r.delta_vs_base, r.delta_low, r.delta_high) if r.method != "reference" else "" for r in unseen_summary.itertuples()]
    unseen_summary.to_csv(REP / "unseen_summary_val.csv", index=False)
    display(unseen_summary.round(4))
    plot_objective(unseen_summary, FIG / "unseen_objective.png", "18 điều kiện chưa thấy, validation")
    show(FIG / "unseen_objective.png")
    plot_levels(unseen_summary, FIG / "unseen_levels.png", "Macro F1 theo mức suy giảm, 6 họ chưa thấy")
    show(FIG / "unseen_levels.png")
else:
    saved = json.loads((REPO_DIR / "reports/unseen/validation_calibration.json").read_text(encoding="utf-8"))
    reference_scores = pd.Series({name: entry["macro_f1"] for name, entry in saved["conditions"].items()}).reindex(unseen_names)
    ta_scores = unseen_frame.set_index("condition")["macro_f1"].reindex(unseen_names)
    unseen_summary = pd.DataFrame({"reference (điểm đã lưu)": reference_scores, "trivialaugment": ta_scores})
    unseen_summary.loc["trung bình 18 điều kiện"] = unseen_summary.mean()
    display(unseen_summary.round(4))
    print("Chưa có checkpoint so sánh, nên chưa có khoảng tin cậy ghép cặp cho phần chưa thấy.")

In [ ]:
unseen_family = family_scores(unseen_labels, unseen_matrices, unseen_names, len(classes)) if paired else None
if paired:
    unseen_family["nhóm"] = [unseen_groups[family] for family in unseen_family.index]
    unseen_family.to_csv(REP / "unseen_family_val.csv")
    display(unseen_family.round(4))
    plot_family_bars(unseen_family.drop(columns="nhóm"), FIG / "unseen_family.png", "Macro F1 theo họ chưa thấy, trung bình ba mức", "Macro F1")
    show(FIG / "unseen_family.png")
    level3 = family_scores(unseen_labels, unseen_matrices, unseen_names, len(classes), level=3)
    plot_family_bars(level3, FIG / "unseen_family_level3.png", "Macro F1 theo họ chưa thấy, mức 3", "Macro F1")
    show(FIG / "unseen_family_level3.png")
    groups = pd.Series({family: unseen_groups[family] for family in unseen_family.index})
    display(unseen_family.drop(columns="nhóm").groupby(groups).mean().round(4))
    comparison = condition_comparison(unseen_labels, unseen_matrices["reference"], unseen_matrices["trivialaugment"], unseen_names, len(classes))
    comparison.to_csv(REP / "unseen_conditions_trivialaugment_vs_reference.csv", index=False)
    plot_delta_heatmap(comparison, FIG / "unseen_delta_ta_vs_ref.png", "Họ chưa thấy: TrivialAugment trừ reference")
    show(FIG / "unseen_delta_ta_vs_ref.png")
    if "degradation_aug" in unseen_matrices:
        comparison = condition_comparison(unseen_labels, unseen_matrices["trivialaugment"], unseen_matrices["degradation_aug"], unseen_names, len(classes))
        comparison.to_csv(REP / "unseen_conditions_degradation_vs_trivialaugment.csv", index=False)
        plot_delta_heatmap(comparison, FIG / "unseen_delta_aug_vs_ta.png", "Họ chưa thấy: augmentation suy giảm trừ TrivialAugment")
        show(FIG / "unseen_delta_aug_vs_ta.png")
else:
    print("Bỏ qua phần theo họ và McNemar vì thiếu dự đoán từng ảnh của model so sánh.")

### Lớp bị mất ở mức 3

Recall từng lớp ở mức 3 của các họ chưa thấy, để xem lỗi dồn vào lớp nào. Ở task 5.5 reference mất nhiều nhất ở `ALLOCARIDARA_ATTACK` khi mờ ống kính và ở `ALGAL_LEAF_SPOT` khi nhiễu màu hoặc tăng bão hòa.

In [ ]:
recall_columns = [column for column in unseen_frame.columns if column.startswith("recall_")]
level3_recall = unseen_frame[unseen_frame["severity"] == 3].set_index("family")[recall_columns]
level3_recall.columns = [column.removeprefix("recall_") for column in recall_columns]
level3_recall.to_csv(REP / "unseen_trivialaugment_recall_level3.csv")
display(level3_recall.round(3))

## 10. Kết luận tự động từ khoảng tin cậy

Các dòng dưới chỉ phát biểu điều khoảng tin cậy ghép cặp cho phép. Kết quả một seed, nên chỉ có giá trị mô tả trên validation, chưa phải kết luận cuối cho P6.1.

In [ ]:
def line(text, a, b, samples_by_method, point_by_method):
    low, high = _interval(samples_by_method[a] - samples_by_method[b])
    delta = point_by_method[a] - point_by_method[b]
    return f"{text}: {delta:+.4f} (CI 95% [{low:+.4f}, {high:+.4f}]) → {verdict(delta, low, high)}"


print("Họ đã thấy (42 điều kiện)")
for text, a, b in pairs:
    print("  " + line(text, a, b, samples, point))
if paired:
    unseen_indices = bootstrap_indices(len(unseen_labels), 1000, 42)
    unseen_samples = {name: objective_samples(unseen_labels, matrix, len(classes), unseen_indices) for name, matrix in unseen_matrices.items()}
    unseen_point = unseen_summary.set_index("method")["objective"]
    print("Họ chưa thấy (18 điều kiện)")
    for text, a, b in pairs:
        if a in unseen_matrices and b in unseen_matrices:
            print("  " + line(text, a, b, unseen_samples, unseen_point))
else:
    print("Họ chưa thấy: chỉ có điểm, không có khoảng tin cậy ghép cặp")

## 11. Lưu kết quả

Chép file kết quả vào `reports/generic_augmentation/` (bản chốt để commit) và nén toàn bộ vào một file. Checkpoint `swin_tiny_trivialaugment_seed42.pt` nằm ở tab Output, trong `outputs/checkpoints/`. Checkpoint không vào Git.

In [ ]:
name = f"{train_cfg['model_name']}_seed{pipeline_cfg['seed']}"
for source in [ROOT / "outputs/results" / f"{name}.json", ROOT / "outputs/results" / f"{name}_confusion_matrix.png", ROOT / "outputs/results" / f"{name}_history.png"]:
    if source.exists():
        shutil.copy(source, REP / source.name)
for source in sorted(FIG.glob("*.png")):
    shutil.copy(source, REP / source.name)
for source in sorted(RES.glob("*.csv")):
    shutil.copy(source, REP / source.name)

archive = Path("/kaggle/working/generic_augmentation_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in (RES, FIG, REP):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(REPO_DIR).as_posix())
    for path in sorted((ROOT / "outputs/results").glob(f"{name}*")):
        bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")
print(f"Checkpoint: {paths['checkpoint']} ({Path(paths['checkpoint']).stat().st_size / 1024**2:.1f} MB)")